In [4]:
from huggingface_hub import login
login()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from keys import HF_TOKEN

adapter_path = "./llama-3-8B-chat-psychotherapist"
base_model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

device = "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16 if device != "cpu" else torch.float32
).to(device)

model = PeftModel.from_pretrained(base_model, adapter_path).to(device)

messages = [
    {"role": "system", "content": "You are a supportive assistant."},
    {"role": "user", "content": "I feel overwhelmed lately. What can I do right now?"}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)

print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct.
401 Client Error. (Request ID: Root=1-69d51516-38660eef31acf6090e708d03;6ed593a9-b386-4dfc-9ec0-87917a13ca44)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Meta-Llama-3-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

In [1]:
from google.colab import userdata
from datasets import load_dataset

userdata.get('HF_TOKEN')
ds = load_dataset("mpingale/mental-health-chat-dataset")
print(ds)

TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.